## Imports

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('/home/ec2-user/multimodal-rag'))

import nest_asyncio
nest_asyncio.apply()

from tqdm.notebook import tqdm
# Allow notebooks to import from scripts/
REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(REPO_ROOT, "scripts"))


import json
import pandas as pd
from pipeline import MMRAGPipeline

## Configurations

In [ ]:
SEED = 19
QUESTION_PATHS = {
    "FinancialReport" : "../data/raw/REAL-MM-RAG_FinReport/test/qas.jsonl",
    "TechReport" : "../data/raw/REAL-MM-RAG_TechReport/test/qas.jsonl",
    "TechSlides" : "../data/raw/REAL-MM-RAG_TechSlides/test/qas.jsonl"
}
RESULTS_PATH = "./results/baseline_ideal_rag_text"
model_id = "Qwen/Qwen3-VL-8B-Instruct"
checkpoint_path = "../Qwen3-VL-8B-Instruct"

## Load Data

In [ ]:
test_indices = {}  # maps dataset name to set of example_index values for the test fold

for name, path in QUESTION_PATHS.items():
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            records.append({
                "example_index": obj["example_index"],
                "question": obj["question"],
                "answer": obj["answer"],
                "intended_img": obj.get("image_path"),
            })
    df = pd.DataFrame(records).set_index("example_index")
    train = df.sample(frac=0.8, random_state=SEED)
    test  = df.drop(train.index)
    test_indices[name] = set(test.index)
    print(f"{name}: {len(train)} train / {len(test)} test")

In [ ]:
VLM_MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"
VLM_CHECKPOINT = os.path.join(REPO_ROOT, "Qwen3-VL-8B-Instruct")

QUESTION_PATHS = {
    "FinancialReport": os.path.join(REPO_ROOT, "data/raw/REAL-MM-RAG_FinReport/test/qas.jsonl"),
    "TechReport":      os.path.join(REPO_ROOT, "data/raw/REAL-MM-RAG_TechReport/test/qas.jsonl"),
    "TechSlides":      os.path.join(REPO_ROOT, "data/raw/REAL-MM-RAG_TechSlides/test/qas.jsonl"),
}

EMBEDDINGS_DIR = os.path.join(REPO_ROOT, "scripts", "embeddings")
RESULTS_DIR    = os.path.join(REPO_ROOT, "experiments", "results", "image_rag")

TOP_K    = 3
SAMPLING = {"max_new_tokens": 4096}

## Initialize Pipeline

In [ ]:
pipeline = MMRAGPipeline(
    mode="image_rag",
    vlm_model_id=VLM_MODEL_ID,
    vlm_checkpoint_path=VLM_CHECKPOINT,
    image_retriever_persist_dir=EMBEDDINGS_DIR,
    top_k=TOP_K,
)

## Ingest Datasets

Index all three test splits into ChromaDB. This only needs to run once — embeddings are persisted to disk and reloaded automatically on subsequent runs.

In [ ]:
for name, qas_path in QUESTION_PATHS.items():
    print(f"\nIngesting {name}...")
    pipeline.ingest(qas_path)

print("\n✅ All datasets ingested.")

## Inference

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

for name, qas_path in QUESTION_PATHS.items():
    output_path = os.path.join(RESULTS_DIR, f"image_rag_{name}.json")
    print(f"\n{'='*60}")
    print(f"Running inference: {name}")
    print(f"{'='*60}")
    pipeline.run_inference(
        qas_jsonl_path=qas_path,
        output_path=output_path,
        example_indices=test_indices[name],
        sampling_params=SAMPLING,
    )

print("\n✅ Inference complete. Results saved to:", RESULTS_DIR)

## Evaluation

In [ ]:
# Run evaluate.py from the repo root to compute BLEU, ROUGE, BERTScore, and LLM-as-judge:
# Need to fix
#   python scripts/evaluate.py experiments/results/image_rag/*.json \
#       --output experiments/results/image_rag/metrics_summary.json
#
# Set OPENAI_API_KEY in your environment to also enable LLM-as-judge scoring.